In [4]:
import pandas as pd

oni = pd.read_csv("../data/oni.ascii.txt", sep=r"\s+")
print(oni.shape)
print(oni.head())
print(oni.tail())
print(oni.dtypes)

(920, 4)
  SEAS    YR  TOTAL  ANOM
0  DJF  1950  25.01 -1.32
1  JFM  1950  25.36 -1.20
2  FMA  1950  25.88 -1.12
3  MAM  1950  26.24 -1.08
4  AMJ  1950  26.35 -1.10
    SEAS    YR  TOTAL  ANOM
915  MAM  2026  28.09  0.46
916  AMJ  2026  28.74  0.95
917  MJJ  2026  29.02  1.39
918  JJA  2026  29.09  1.80
919  JAS  2026  29.12  2.16
SEAS         str
YR         int64
TOTAL    float64
ANOM     float64
dtype: object


In [5]:
with open("../data/soi_raw.txt") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i > 20:
            break

(STAND TAHITI - STAND DARWIN)  SEA LEVEL PRESS
                        ANOMALY

YEAR   JAN   FEB   MAR   APR   MAY   JUN   JUL   AUG   SEP   OCT   NOV   DEC
1951   2.5   1.5  -0.2  -0.5  -1.1   0.3  -1.7  -0.4  -1.8  -1.6  -1.3  -1.2
1952  -1.5  -1.0   0.9  -0.4   1.2   1.2   0.8   0.1  -0.4   0.6   0.0  -2.0
1953   0.5  -0.8  -0.3   0.3  -2.8   0.2  -0.0  -2.0  -2.1   0.1  -0.5  -0.8
1954   1.1  -0.5   0.4   1.1   0.8   0.2   0.7   1.8   0.3   0.4   0.2   2.3
1955  -0.9   3.1   1.1  -0.2   1.7   2.2   2.6   2.4   2.2   2.5   2.0   1.6
1956   2.2   2.7   2.2   1.5   2.3   1.8   1.8   2.0   0.1   2.9   0.2   1.8
1957   1.0  -0.1   0.3   0.4  -1.1   0.3   0.4  -0.8  -1.5   0.1  -1.6  -0.5
1958  -3.1  -0.8   0.4   0.6  -0.8   0.5   0.7   1.5  -0.5   0.1  -0.7  -1.0
1959  -1.5  -2.3   2.1   0.7   0.8  -0.2  -0.4  -0.2   0.0   0.8   1.5   1.5
1960   0.2   0.2   1.7   1.3   0.9   0.2   0.8   1.3   1.1   0.2   0.9   1.3
1961  -0.4   1.5  -3.0   1.3   0.5   0.2   0.4   0.2   0.1  -0.5   0.9   

In [7]:
import pandas as pd
import numpy as np

months = ["JAN", "FEB", "MARCH", "APRIL", "MAY", "JUN", "JULY", "AUG", "SEP", "OCT", "NOV", "DEC"]

soi_wide = pd.read_fwf(
    "../data/soi_raw.txt",
    skiprows=4,          
    nrows=80,            
    widths=[4] + [6]*12, 
    header=None,
    names=["YEAR"] + months,
)

soi_wide = soi_wide.replace(-999.9, np.nan)
print(soi_wide.shape)
print(soi_wide.tail(6))
print(soi_wide.isna().sum().sum())

(80, 13)
    YEAR  JAN  FEB  MARCH  APRIL  MAY  JUN  JULY  AUG  SEP  OCT  NOV  DEC
74  2025  0.3  0.9    2.8    0.9  0.7  0.5   1.0  0.7  0.1  1.9  1.8 -0.0
75  2026  1.8  2.4    2.0   -1.1 -1.5 -2.4  -4.0 -1.8 -3.3  NaN  NaN  NaN
76  2027  NaN  NaN    NaN    NaN  NaN  NaN   NaN  NaN  NaN  NaN  NaN  NaN
77  2028  NaN  NaN    NaN    NaN  NaN  NaN   NaN  NaN  NaN  NaN  NaN  NaN
78  2029  NaN  NaN    NaN    NaN  NaN  NaN   NaN  NaN  NaN  NaN  NaN  NaN
79  2030  NaN  NaN    NaN    NaN  NaN  NaN   NaN  NaN  NaN  NaN  NaN  NaN
51


## Step 1: Data Inspections

**ONI**
- 920 rows, 4 columns (SEAS, YR, TOTAL, ANOM)
- Covers DJF 1950 to JAS 2026
- SEAS is a 3-month season code (DJF, JFM, FMA...). YR refers to the middle month.
- Data types look fine. 
- Latest values are strongly positive (ANOM 2.16 in JAS 2026), above the +0.5 El Niño threshold.

**SOI**
- Header text above the data
- Wide format: one column per month, one row per year
- File holds two blocks: anomaly and standardized
- Missing values are coded as -999.9 (Oct 2026 and all of 2027 to 2030)
- Some numbers run together with no space (for example -3.3-999.9), so it needs fixed-width reading
- Starts in 1951, while ONI starts in 1950
- A few -0.0 values

-The ongoing 2026 event is labeled El Niño even though it has not yet reached 5 months, since it is still developing.

